# Fabric Data Agent — Sales Pipeline Demo

**Run this notebook inside a Fabric workspace.** It will:
1. Install `fabric-data-agent-sdk`.
2. Create a Data Agent named `sales_pipeline_agent` (idempotent — re-runs update).
3. Add the demo Lakehouse and Warehouse as data sources.
4. Add AI instructions + few-shot example queries.
5. Publish.
6. Smoke-test with a question.

**Prereqs**
- Workspace attached to a paid Fabric capacity (F2+) — you said capacity is started.
- Lakehouse named `lh_silver` (or change `LAKEHOUSE_NAME` below).
- Warehouse named `wh_gold` (or change `WAREHOUSE_NAME`; set to `None` to skip).
- You are signed into Fabric (the notebook runs as you).

**How to import this notebook into Fabric**
1. Fabric portal → your workspace → **+ New → Import notebook → Upload from this computer**.
2. Pick this `.ipynb` file.
3. Attach a Lakehouse (top of notebook) — pick `lh_silver`.
4. **Run all**.

In [ ]:
%pip install -q fabric-data-agent-sdk

In [ ]:
# === Config ===
AGENT_NAME      = "sales_pipeline_agent"
LAKEHOUSE_NAME  = "lh_silver"      # change if different
WAREHOUSE_NAME  = None        # set to None to skip

AGENT_INSTRUCTIONS = (
    "You are a sales analytics assistant for Contoso. Answer questions about leads, "
    "opportunities, pipeline coverage, deal stages, and quota attainment. "
    "Always express monetary amounts in USD. When the user says 'pipeline', they mean the "
    "sum of open opportunity amounts weighted by stage probability. "
    "Prefer recent data (current and previous fiscal quarter) unless the user specifies a window. "
    "If asked about a salesperson, match by full name (case-insensitive)."
)

# Datasource-level guidance for the lakehouse
LAKEHOUSE_NOTES = (
    "Tables of interest: opportunities, leads, accounts, sales_reps, stage_probability. "
    "Join opportunities.account_id = accounts.account_id and opportunities.owner_id = sales_reps.rep_id. "
    "Open opportunities have stage NOT IN ('ClosedWon','ClosedLost'). "
    "Lead score >= 70 means 'qualified'."
)

# Few-shot examples (NL question -> SQL). Use lakehouse SQL syntax.
LAKEHOUSE_FEWSHOTS = {
    "What is total open pipeline by stage this quarter?":
        "SELECT o.stage, SUM(o.amount * sp.probability) AS weighted_pipeline "
        "FROM opportunities o JOIN stage_probability sp ON o.stage = sp.stage "
        "WHERE o.stage NOT IN ('ClosedWon','ClosedLost') "
        "  AND o.close_date >= date_trunc('quarter', current_date) "
        "GROUP BY o.stage ORDER BY weighted_pipeline DESC",

    "How many qualified leads did each rep generate last month?":
        "SELECT r.full_name, COUNT(*) AS qualified_leads "
        "FROM leads l JOIN sales_reps r ON l.owner_id = r.rep_id "
        "WHERE l.lead_score >= 70 "
        "  AND l.created_date >= date_trunc('month', current_date - INTERVAL 1 MONTH) "
        "  AND l.created_date <  date_trunc('month', current_date) "
        "GROUP BY r.full_name ORDER BY qualified_leads DESC",

    "Top 5 accounts by open opportunity value":
        "SELECT a.account_name, SUM(o.amount) AS open_value "
        "FROM opportunities o JOIN accounts a ON o.account_id = a.account_id "
        "WHERE o.stage NOT IN ('ClosedWon','ClosedLost') "
        "GROUP BY a.account_name ORDER BY open_value DESC LIMIT 5",
}

WAREHOUSE_NOTES = (
    "This is the gold curated warehouse. Use fact_sales for closed-won revenue and "
    "dim_rep / dim_account for descriptive attributes. Quotas are in fact_quota by quarter."
)

WAREHOUSE_FEWSHOTS = {
    "Who hit quota last quarter?":
        "SELECT r.full_name, SUM(s.amount) AS booked, q.quota_amount, "
        "       SUM(s.amount) * 1.0 / NULLIF(q.quota_amount,0) AS attainment "
        "FROM fact_sales s JOIN dim_rep r ON s.rep_id = r.rep_id "
        "JOIN fact_quota q ON q.rep_id = r.rep_id AND q.quarter = s.quarter "
        "WHERE s.quarter = DATEADD(quarter, -1, GETDATE()) "
        "GROUP BY r.full_name, q.quota_amount HAVING attainment >= 1.0 "
        "ORDER BY attainment DESC",
}

In [ ]:
# === Convert seeded CSVs in Files/sales/ to Delta tables in Tables/ ===
# Uses abfss:// paths so no UI lakehouse attachment is needed (headless-friendly).
# WORKSPACE_NAME and LAKEHOUSE_NAME are injected by the trigger script.
try:
    WORKSPACE_NAME  # injected
except NameError:
    raise RuntimeError('WORKSPACE_NAME parameter missing; run via trigger_data_agent_notebook.py')

from pyspark.sql.types import StringType
_base = f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_NAME}.Lakehouse"
_tables = ['accounts','contacts','sales_reps','leads','opportunities',
           'stage_probability','fact_quota','fact_sales']
for _t in _tables:
    _src = f"{_base}/Files/sales/{_t}.csv"
    _dst = f"{_base}/Tables/{_t}"
    df = spark.read.option('header', True).option('inferSchema', True).csv(_src)
    df.write.mode('overwrite').format('delta').save(_dst)
    print(f'  wrote Tables/{_t}  ({df.count()} rows)')


In [ ]:
from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
)

# Idempotent create-or-open
try:
    agent = FabricDataAgentManagement(AGENT_NAME)
    print(f"Opened existing data agent: {AGENT_NAME}")
except Exception:
    agent = create_data_agent(AGENT_NAME)
    print(f"Created new data agent: {AGENT_NAME}")

agent.update_configuration(instructions=AGENT_INSTRUCTIONS)
agent.get_configuration()

In [ ]:
# Add Lakehouse datasource
existing = {d.name if hasattr(d, 'name') else str(d): d for d in agent.get_datasources()}
if LAKEHOUSE_NAME not in existing:
    lh = agent.add_datasource(LAKEHOUSE_NAME, type="lakehouse")
    print(f"Added lakehouse: {LAKEHOUSE_NAME}")
else:
    lh = existing[LAKEHOUSE_NAME]
    print(f"Lakehouse already attached: {LAKEHOUSE_NAME}")

lh.update_configuration(instructions=LAKEHOUSE_NOTES)

# Select tables — pretty_print() shows what's available; we select common sales tables.
for tbl in ("opportunities", "leads", "accounts", "sales_reps", "stage_probability"):
    try:
        lh.select("dbo", tbl)
        print(f"  selected dbo.{tbl}")
    except Exception as e:
        print(f"  (skip dbo.{tbl}: {e})")

lh.add_fewshots(LAKEHOUSE_FEWSHOTS)
print("Lakehouse few-shots:", len(lh.get_fewshots()))

In [ ]:
# Add Warehouse datasource (optional)
if WAREHOUSE_NAME:
    existing = {d.name if hasattr(d, 'name') else str(d): d for d in agent.get_datasources()}
    if WAREHOUSE_NAME not in existing:
        wh = agent.add_datasource(WAREHOUSE_NAME, type="warehouse")
        print(f"Added warehouse: {WAREHOUSE_NAME}")
    else:
        wh = existing[WAREHOUSE_NAME]
        print(f"Warehouse already attached: {WAREHOUSE_NAME}")
    wh.update_configuration(instructions=WAREHOUSE_NOTES)
    for tbl in ("fact_sales", "fact_quota", "dim_rep", "dim_account"):
        try:
            wh.select("dbo", tbl)
            print(f"  selected dbo.{tbl}")
        except Exception as e:
            print(f"  (skip dbo.{tbl}: {e})")
    wh.add_fewshots(WAREHOUSE_FEWSHOTS)
    print("Warehouse few-shots:", len(wh.get_fewshots()))
else:
    print("No warehouse configured.")

In [ ]:
# Publish the agent
agent.publish()
print("Published.")
print("Datasources:", agent.get_datasources())

In [ ]:
# Smoke test using the OpenAI Assistants API surface the SDK exposes
from fabric.dataagent.client import FabricOpenAI
import time

client = FabricOpenAI(artifact_name=AGENT_NAME)
assistant = client.beta.assistants.create(model="not-used")
thread = client.beta.threads.create()
client.beta.threads.messages.create(thread_id=thread.id, role="user",
    content="Top 5 accounts by open opportunity value")
run = client.beta.threads.runs.create(thread_id=thread.id, assistant_id=assistant.id)

for _ in range(60):
    run = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    if run.status in ("completed", "failed", "cancelled", "expired"):
        break
    time.sleep(3)

print("run status:", run.status)
for m in client.beta.threads.messages.list(thread_id=thread.id, order="asc").data:
    print(f"--- {m.role} ---")
    for c in m.content:
        if c.type == "text":
            print(c.text.value)

## Get the Published URL for Copilot Studio
Open the Data Agent in Fabric → Settings → copy **Published URL**.

In Copilot Studio:
1. Your copilot → **Tools → + Add tool → Connector → Microsoft Fabric Data Agent** (preview).
2. Paste the Published URL, sign in (OAuth).
3. Name the tool `Sales Pipeline Data Agent`, description: *"Use for questions about Contoso leads, opportunities, pipeline coverage, deal stages, quota attainment."*
4. Add to a topic or let the orchestrator route. Test → **Publish**.